# Lab 2: Pandas for Cat and Dog Faces

In this notebook, you will treat the cat-and-dog-faces dataset as a **table** and practice core Pandas operations.

This lab is self-contained and does **not** depend on Lab 1. You will build the metadata table yourself from the folder structure, save it, load it with Pandas, and analyze it.

**Questions in this lab**

1. Build metadata from folders
2. Load the saved metadata with Pandas
3. Inspect the metadata table
4. Count each split by class
5. Audit metadata quality
6. Add analysis columns
7. Visualize split balance
8. Create a balanced sample by split and class


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from lab_utils.visualization import (
    plot_class_balance,
    plot_numeric_distribution,
)

def find_project_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "data" / "cats_dogs_faces_small").exists():
            return candidate
    return Path.cwd().resolve()

PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data" / "cats_dogs_faces_small"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

STUDENT_ID = 10422021  # Replace with your own student ID.
SEED = int(STUDENT_ID)
GENERATED_METADATA_PATH = ARTIFACT_DIR / f"lab2_faces_metadata_{STUDENT_ID}.csv"

expected = [
    DATA_ROOT / "train" / "cat",
    DATA_ROOT / "train" / "dog",
    DATA_ROOT / "val" / "cat",
    DATA_ROOT / "val" / "dog",
    DATA_ROOT / "test" / "cat",
    DATA_ROOT / "test" / "dog",
]
if not all(path.exists() for path in expected):
    raise FileNotFoundError(
        "Dataset not found. Run `uv run python scripts/download_animal_faces.py --force` first."
    )

print(f"Student ID seed: {STUDENT_ID}")
print(f"Dataset root: {DATA_ROOT}")
print(f"Generated metadata path: {GENERATED_METADATA_PATH}")


## Question 1: Build metadata from the folder structure

Break the metadata-building task into smaller helper functions, then call them from `build_metadata_from_folders(...)`.

Build a dataframe with at least these columns:

- `filepath`
- `label`
- `split`
- `width`
- `height`
- `mean_intensity`

Suggested subtasks:

1. list image paths for one split and one label
2. inspect one image file to get width, height, and mean intensity
3. convert one image path into one metadata row
4. combine all rows into one dataframe


In [ ]:
def list_image_paths_for_group(data_root: Path, split: str, label: str) -> list[Path]:
    # TODO: return all image paths for one split/label pair.
    raise NotImplementedError("List the image paths for one split and one label.")


def inspect_image_file(path: Path) -> tuple[int, int, float]:
    # TODO: open one image and return width, height, and mean intensity.
    raise NotImplementedError("Inspect one image file.")


def make_metadata_row(path: Path, data_root: Path, split: str, label: str) -> dict[str, object]:
    # TODO: turn one image path into one metadata row dictionary.
    raise NotImplementedError("Create one metadata row from one image path.")


def build_metadata_from_folders(data_root: Path) -> pd.DataFrame:
    split_names = ("train", "val", "test")
    label_names = ("cat", "dog")
    rows = []

    for split in split_names:
        for label in label_names:
            image_paths = list_image_paths_for_group(data_root, split, label)
            rows.extend([make_metadata_row(path, data_root, split, label) for path in image_paths])

    return pd.DataFrame(rows).sort_values(["split", "label", "filepath"]).reset_index(drop=True)


folder_df = build_metadata_from_folders(DATA_ROOT)
print("metadata shape:", folder_df.shape)
display(folder_df.head())

folder_df.to_csv(GENERATED_METADATA_PATH, index=False)
print(f"Saved generated metadata to: {GENERATED_METADATA_PATH}")


## Question 2: Load the saved metadata with Pandas

Use `pd.read_csv(...)` inside a small helper function.

This keeps the workflow realistic: build a table once, then reload it as a dataframe for analysis.


In [ ]:
def load_metadata_table(csv_path: Path) -> pd.DataFrame:
    # TODO: load the csv with Pandas and return the dataframe.
    raise NotImplementedError("Load the saved metadata table with Pandas.")


df = load_metadata_table(GENERATED_METADATA_PATH)
print("loaded shape:", df.shape)
display(df.head())


## Question 3: Inspect the metadata table

Summarize the dataset with Pandas operations. Return:

- number of rows
- column names
- class counts
- split counts


In [ ]:
def summarize_metadata(frame: pd.DataFrame) -> dict[str, object]:
    # TODO: return a dictionary with rows, columns, class_counts, and split_counts.
    """
    Example output:
    output_dictionary = {
        "rows":,
        "columns":,
        "class_counts":,
        "split_counts":,
    }
    """
    raise NotImplementedError("Summarize the metadata table with Pandas.")


metadata_summary = summarize_metadata(df)
print("Rows:", metadata_summary["rows"])
print("Columns:", metadata_summary["columns"])
print("\nClass counts:")
print(metadata_summary["class_counts"])
print("\nSplit counts:")
print(metadata_summary["split_counts"])


## Question 4: Count each split by class

Build a table that answers questions like:

- how many dog images are in `train`?
- how many dog images are in `val`?
- how many dog images are in `test`?
- and the same for cats

Hint:

Use `groupby(["label", "split"]).size().unstack(fill_value=0)` inside your helper function.


In [ ]:
def build_label_split_table(frame: pd.DataFrame) -> pd.DataFrame:
    # TODO: return a table with labels as rows and splits as columns.
    raise NotImplementedError("Build the label-by-split count table.")


label_split_table = build_label_split_table(df)
display(label_split_table)


## Question 5: Audit metadata quality

Check for:

- missing values
- duplicated file paths
- unexpected labels
- non-positive image sizes


In [ ]:
def audit_metadata(frame: pd.DataFrame) -> dict[str, object]:
    # TODO: return a dictionary with missing_values, duplicate_filepaths, bad_labels, and non_positive_sizes.
    raise NotImplementedError("Audit the metadata table.")


audit_report = audit_metadata(df)
audit_report


## Question 6: Add analysis columns

Create a helper function that returns a copy of the dataframe with these new columns:

- `pixel_count = width * height`
- `aspect_ratio = width / height`
- `brightness_band` using `pd.qcut(...)` with 4 bins
- `size_bucket` using labels like `small`, `medium`, and `large`


In [ ]:
def add_analysis_columns(frame: pd.DataFrame) -> pd.DataFrame:
    # TODO: return a copy of frame with the requested analysis columns added.
    raise NotImplementedError("Create the analysis columns with Pandas.")


analysis_df = add_analysis_columns(df)
display(analysis_df.head())


## Question 7: Visualize the data split

Build a split-balance table that counts cats and dogs inside each split, then visualize it.

Hint:

Use `groupby(["split", "label"]).size().unstack(fill_value=0)` inside your helper function.


In [ ]:
def build_split_balance_table(frame: pd.DataFrame) -> pd.DataFrame:
    # TODO: return a split-by-label count table.
    raise NotImplementedError("Build the split balance summary table.")


split_balance = build_split_balance_table(analysis_df)
display(split_balance)

plot_class_balance(analysis_df, title="Class balance by split")


## Question 8: Create a balanced sample by split and class

Build a smaller dataframe for quick experiments.

Requirement:

Sample the same number of images for each `(split, label)` group so the sample stays balanced across both the data split and the class.

Suggested property to preserve:

- every split should still contain both cats and dogs
- each `(split, label)` group should contribute the same number of rows

Hint:

Use `groupby(["split", "label"])` and sample within each group with a fixed random seed.


In [ ]:
def sample_balanced_by_split_and_label(frame: pd.DataFrame, n_per_group: int, seed: int) -> pd.DataFrame:
    # TODO: sample the same number of rows from each (split, label) group and return the combined dataframe.
    raise NotImplementedError("Build a balanced sample across split and label groups.")


sample_size_per_group = 5
sampled_df = sample_balanced_by_split_and_label(analysis_df, n_per_group=sample_size_per_group, seed=SEED)
print("sampled shape:", sampled_df.shape)
display(sampled_df.head())

sampled_balance = sampled_df.groupby(["split", "label"]).size().unstack(fill_value=0)
display(sampled_balance)

plot_class_balance(sampled_df, title="Balanced sampled subset by split")


## Reflection

Write short answers to these questions:

1. Is the dataset balanced across classes and splits?
2. Did your sampled subset stay balanced across splits and classes?
3. Why is grouped sampling useful before training a model?
4. Which Pandas operation felt most useful in this lab: `groupby`, `value_counts`, `qcut`, or sampling within groups? Why?
